# 📘 CityAI – Cerca de textos a les subvencions públiques

Aquest notebook utilitza dades obertes de l’Ajuntament de Barcelona per practicar conceptes d’algorísmica. En concret es treballaran els algorismes de cerca de text exacte i altres algorismes de manipulació de cadenes.

## 🧠 Objectius

Aprofundir en els algorismes vistos a teoria, i veure la seva aplicació en un cas real.

## ✍️ Exercici 1: LLegir les dades del fitxer i guardar-les en una llista

L'Ajuntament de Barcelona guarda i publica les dades de les subvencions que concedeix en un fitxer públic. Aquest fitxer té les següents dades:
+ L'entitat municipal que ha concedit la subvenció
+ L'any en què s'ha concedit la subvenció
+ La tipologia de subvenció
+ L'objecte de la subvenció
+ L'import sol.licitat

*Nota*: Hem reduit el fitxer i simplificat les dades per facilitar  treballar-hi. Per aixo treballarem amb el fitxer SubvencionsReduitNetejat.csv

Escriu una funció que llegeixi les dades de SubvencionsReduitNetejat.csv i les guardi en una llista de llistes.

In [6]:
def llegir_dades(nom_fitxer:str)->list[list[any]]:
    """
    Aquesta funció llegeix les dades dels fitxers de subvencions de l'Ajuntament 
    de Barcelona i retorna una llista de llistes
    
    Parameters
    ----------
    nom_fitxer:str
    
    Returns
    -------
    dades: list[list[any]]
    """
    dades = []
    with open("SubvencionsReduitNetejat.csv", "r") as file:
        for line in file:
            dada = line.strip().split(";")
            dada[1] = int(dada[1])
            dada[-1] = int(dada[-1])

            dades.append(dada)

    return dades

In [83]:
# ✅ Asserts públics
dades = llegir_dades("SubvencionsReduitNetejat.csv")
assert(dades[5]==['Ajuntament de Barcelona', 2022, 'Convocatoria 2022 Impuls socioeconomic del territori IMPULSEM', 'L estiu Cooperatiu de Sant Marti', 400000])
assert(dades[159]==['Ajuntament de Barcelona', 2017, 'CONVOCATORIA GENERAL DE SUBVENCIONS 2017  SANT ANDREU', 'CULTURA INTERBARRIAL', 12000])

## ✍️ Exercici 2 – Subvencions d'un tema.

Hi ha alguna subvenció relacionada amb els gegants? (cerca exacta per Horspool)

Aplica l'algorisme de Horspool de teoria per buscar alguns temes específics dins dels objectes de  les subvencions. Els temes s'han de trobar tant si estan en majúscula com en minúscula.

Prova de trobar "gegant", "salud", "cultura", "teatre", "infancia"

Si els trobes retorna la fila de dades que els conté i destaca la seva aparació amb ** davant i ** darrera del text trobat.

Per ex. buscar_patro(dades,"gegant") ha de retornar:

```python
[['Ajuntament de Barcelona',
  2019,
  'CONVOCATORIA GENERAL DE SUBVENCIONS 2019. HORTA GUINARDO',
  'Una **gegant**a  Carmela  i els seus cap grossos.',
  46700]]
```

In [ ]:
def BoyerMooreHorspool(patro:str, text:str)->int:
    """ Retorna l'index del primer patro en text. Si no es troba, retorna -1
    Parameters:
        patro, text: str
            cadenes de text que només conté caràcteres i espais
    """
    # create dict_skip
    dict_skip = {}
    for i in range(len(patro) - 1):
        dict_skip[patro[i]] = len(patro) - i - 1

    # search
    i = len(patro) - 1  # puntero de text
    while i < len(text):
        # backtrace
        pi = len(patro) - 1
        ti = i
        while patro[pi] == text[ti]:
            if pi == 0: return i - len(patro) + 1  # found

            pi -= 1
            ti -= 1
        
        if text[i] in dict_skip:
            i += dict_skip[text[i]]
        else:
            i += len(patro)
    
    return -1  # not found

In [85]:
def buscar_patro(dades: list[list[any]], patro: str) -> list[list[any]]:
    DADES_A_BUSCAR = (0, 2, 3)
    lower_patro = patro.lower()
    dades_amb_patro = []

    for dada in dades:
        _dada = dada.copy()
        for i in DADES_A_BUSCAR:
            text = _dada[i]
            lower_text = text.lower()
            index_patro = BoyerMooreHorspool(lower_patro, lower_text)

            if index_patro != -1:
                # insert **
                _dada[i] = text[:index_patro] + "**" + text[index_patro : index_patro + len(patro)] + "**" + text[index_patro + len(patro):]
                dades_amb_patro.append(_dada)
    
    return dades_amb_patro

In [60]:
BoyerMooreHorspool("wahuuuuu", "wauuuauuhwhhh  huuwahuuuuuwawuhuhuhhuu")

18

In [26]:
buscar_patro(dades,"gegant")

[['Ajuntament de Barcelona',
  2019,
  'CONVOCATORIA GENERAL DE SUBVENCIONS 2019  HORTA GUINARDO',
  'Una **gegant**a  Carmela  i els seus cap grossos ',
  46700]]

In [88]:
# ✅ Asserts públics
assert(buscar_patro(dades,"infancia")) == [['Ajuntament de Barcelona',
  2017,
  'AJUTS',
  'Ajut del programa Promocio i participacio **infancia**',
  52290],
 ['Ajuntament de Barcelona',
  2017,
  'AJUTS',
  'Ajut del programa Promocio i participacio **infancia**',
  53070],
 ['Ajuntament de Barcelona',
  2018,
  'AJUTS',
  'Ajut del programa Promocio i participacio **infancia**',
  47850],
 ['Ajuntament de Barcelona',
  2017,
  'AJUTS',
  'Ajut del programa Promocio i participacio **infancia**',
  3960]]

## ✍️ Exercici 3 – Abreviació d'entitats.

Hi ha tantes subvencions que l'ajuntament ha decidit guardar els noms de les entitats que concedeixen els ajuts de forma abreujada, guardant només les seves inicials. Per exemple: Institut Municipal Barcelona Esports es guardarà com IMBE

Escriu una funció que a partir d'una frase sense punts, comes, accents, números ni cap signe de puntuació, imprimeixi l'acrònim corresponent.

In [91]:
def acronim(frase):
    """
    Aquesta funció retorna l'acrònim d'una frase.
    
    Parameters
    ----------
    frase: string
    
    Returns
    -------
    acronim: string
    """
    acronim = ""
    for word in frase.split():
        acronim += word[0].upper()

    return acronim

In [93]:
# ✅ Asserts públics
assert(acronim("Institut Municipal Barcelona Esports")) == "IMBE"
assert(acronim("Institut Municipal Barcelona Esports")) == "IMBE"

## ✍️ Exercici 4 – Alfabet d'aviació

Com que ara hi ha tantes sigles, sovint hi ha confusions per telèfon, i els responsables de l'ajuntament han adoptat l'alfabet de l'aviació per a comunicar-se.

Alfabet d'aviació:
|   |   |   |   |   |
|---|---|---|---|---|
| A | B | C | D | E |
| Alpha | Bravo | Charlie | Delta | Echo |
| F | G | H | I | J |
| Foxtrot | Golf | Hotel | India | Juliet |
| K | L | M | N | O |
| Kilo | Lima | Mike | November | Oscar |
| P | Q | R | S | T |
| Papa | Quebec | Romeo | Sierra | Tango |
| U | V | W | X | Y |
| Uniform | Victor | Whiskey | X-ray | Yankee |
| Z |   |   |   |   |
| Zulu |   |   |   |   |

Escriu una funció que, usant un diccionari, converteixi una cadena de lletres a l'alfabet d'aviació, generant una llista de sortida.

Per exemple: IMBE generarà la llista [India, Mike, Bravo, Echo]

In [94]:
def aviacio(cadena):
    """
    Aquesta funció converteix una cadena d'entrada a l'alfabet fonètic.
    
    Parameters
    ----------
    cadena: string
    
    Returns
    -------
    traduccio: string
    """
    diccionari = {
        "A": "Alpha",
        "B": "Bravo",
        "C": "Charlie",
        "D": "Delta",
        "E": "Echo",
        "F": "Foxtrot",
        "G": "Golf",
        "H": "Hotel",
        "I": "India",
        "J": "Juliet",
        "K": "Kilo",
        "L": "Lima",
        "M": "Mike",
        "N": "November",
        "O": "Oscar",
        "P": "Papa",
        "Q": "Quebec",
        "R": "Romeo",
        "S": "Sierra",
        "T": "Tango",
        "U": "Uniform",
        "V": "Victor",
        "W": "Whiskey",
        "X": "X-ray",
        "Y": "Yankee",
        "Z": "Zulu"
    }

    traduccio = []
    for ch in cadena:
        traduccio.append(diccionari[ch.upper()])

    return traduccio

In [95]:
# ✅ Asserts públics
assert aviacio('YtH') == ['Yankee', 'Tango', 'Hotel']